### V0.1

1) age      :int64	    顧客の年齢。
2) job      :object	職業（admin, blue-collar, technician など）。
4) education:object	学歴（primary, secondary, tertiary など）
13) campaign :int64     このキャンペーン（勧誘）期間中の接触回数
16) poutcome :object	前回のキャンペーン（勧誘）の成果（success(契約してくれた), failure(断られた) など）。

# ライブラリ

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# 学習モデル作成

In [41]:
df = pd.read_pickle('data.pkl')
df

,id,age,job,education,campaign,poutcome,y
0,0,42,technician,secondary,3,unknown,0
1,1,38,blue-collar,secondary,1,unknown,0
2,2,36,blue-collar,secondary,2,unknown,0
3,3,27,student,secondary,2,unknown,0
4,4,26,technician,secondary,1,unknown,1
...,...,...,...,...,...,...,...
749995,749995,29,services,secondary,2,unknown,1
749996,749996,69,retired,tertiary,1,unknown,0
749997,749997,50,blue-collar,secondary,1,unknown,0
749998,749998,32,technician,secondary,6,unknown,0


In [42]:
y = df["y"]
x = df.drop(columns=["y", "id"])
x = pd.get_dummies(x, drop_first=True)

In [43]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

In [44]:
train_data = lgb.Dataset(x_train,label=y_train)
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

In [45]:
model = lgb.train(params, train_data,num_boost_round=100)

[LightGBM] [Info] Number of positive: 72283, number of negative: 527717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005625 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 155
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120472 -> initscore=-1.987971
[LightGBM] [Info] Start training from score -1.987971


In [46]:
y_pred_proba = model.predict(x_test)
y_pred_proba

array([0.06112282, 0.78977886, 0.06169181, ..., 0.06664939, 0.0725132 ,
       0.09237789], shape=(150000,))

In [47]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.7164416803000848


# テスト、提出

In [48]:
df_test = pd.read_pickle('test.pkl')
df_test

,id,age,job,education,campaign,poutcome
0,750000,32,blue-collar,secondary,1,unknown
1,750001,44,management,tertiary,2,unknown
2,750002,36,self-employed,primary,2,unknown
3,750003,58,blue-collar,secondary,1,unknown
4,750004,28,technician,secondary,1,unknown
...,...,...,...,...,...,...
249995,999995,43,management,tertiary,2,unknown
249996,999996,40,services,unknown,1,failure
249997,999997,63,retired,primary,1,success
249998,999998,50,blue-collar,primary,2,unknown


In [49]:
submit_x = df_test.drop(columns=["id"])
submit_x = pd.get_dummies(submit_x, drop_first=True)
submit_x = submit_x.reindex(columns=x.columns, fill_value=0)

In [50]:
submit_x

,age,campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown
0,32,1,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
1,44,2,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True
2,36,2,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
3,58,1,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
4,28,1,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,43,2,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True
249996,40,1,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False
249997,63,1,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
249998,50,2,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True


In [51]:
test_pred = model.predict(submit_x)
test_pred

array([0.06529883, 0.10079895, 0.0601608 , ..., 0.79371316, 0.04272642,
       0.3016291 ], shape=(250000,))

In [52]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})
submission.to_csv('submission.csv', index=False)